In [1]:
!pip -q install streamlit plotly pandas numpy

print("STEP 1 COMPLETE: Required packages installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 74.7 MB/s eta 0:00:00
STEP 1 COMPLETE: Required packages installed.


In [2]:
from google.colab import files
import os

uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No file was uploaded.")

uploaded_name = next(iter(uploaded.keys()))
target_name = "Atlantic_South_Korea.csv"

if uploaded_name != target_name:
    os.rename(uploaded_name, target_name)

print("STEP 2 COMPLETE")
print("Dataset:", target_name)
print("File size:", os.path.getsize(target_name), "bytes")

Saving Atlantic_South_Korea.csv to Atlantic_South_Korea.csv
STEP 2 COMPLETE
Dataset: Atlantic_South_Korea.csv
File size: 3635211 bytes


In [3]:
import pandas as pd
import numpy as np

raw = pd.read_csv("Atlantic_South_Korea.csv")

print("STEP 3 COMPLETE")
print("Shape:", raw.shape)

print("\nColumns:")
print(raw.columns.tolist())

print("\nFirst 5 rows:")
display(raw.head())

STEP 3 COMPLETE
Shape: (27800, 10)

Columns:
['date', 'position', 'song', 'artist', 'popularity', 'duration_ms', 'album_type', 'total_tracks', 'is_explicit', 'album_cover_url']

First 5 rows:


,date,position,song,artist,popularity,duration_ms,album_type,total_tracks,is_explicit,album_cover_url
0,18-05-2024,1,Like Crazy,Jimin,93,212241,single,6,False,https://i.scdn.co/image/ab67616d0000b2732b4607...
1,18-05-2024,2,UNFORGIVEN (feat. Nile Rodgers),LE SSERAFIM,87,182148,album,13,False,https://i.scdn.co/image/ab67616d0000b273d71fd7...
2,18-05-2024,3,Spicy,aespa,81,197040,single,6,False,https://i.scdn.co/image/ab67616d0000b27304878a...
3,18-05-2024,4,I AM,IVE,89,183853,album,11,False,https://i.scdn.co/image/ab67616d0000b27325ef3c...
4,18-05-2024,5,Queencard,(G)I-DLE,71,161240,single,6,False,https://i.scdn.co/image/ab67616d0000b27382dd24...


In [4]:
required_columns = [
    "date",
    "position",
    "song",
    "artist",
    "popularity",
    "duration_ms",
    "album_type",
    "total_tracks",
    "is_explicit",
    "album_cover_url"
]

missing = [c for c in required_columns if c not in raw.columns]

print("STEP 4 — COLUMN VALIDATION")

if missing:
    print("Missing columns:", missing)
    raise ValueError("Required columns are missing.")
else:
    print("All required columns are present.")

check = raw.copy()
check["date"] = pd.to_datetime(check["date"], errors="coerce")

print("\nMissing values:")
display(
    check[required_columns]
    .isna()
    .sum()
    .to_frame("missing_values")
)

print("\nRows per date — unusual counts:")

daily_counts_raw = (
    check.groupby("date")
    .size()
    .reset_index(name="records")
)

display(
    daily_counts_raw[
        daily_counts_raw["records"] != 50
    ].head(20)
)

print("\nUnique dates:", check["date"].nunique())

print("STEP 4 COMPLETE")

STEP 4 — COLUMN VALIDATION
All required columns are present.

Missing values:


/tmp/ipykernel_1460/333011661.py:25: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  check["date"] = pd.to_datetime(check["date"], errors="coerce")


,missing_values
date,0
position,0
song,0
artist,0
popularity,0
duration_ms,0
album_type,0
total_tracks,0
is_explicit,0
album_cover_url,0



Rows per date — unusual counts:


,date,records
287,2025-03-01,100



Unique dates: 555
STEP 4 COMPLETE


In [5]:
df = raw[required_columns].copy()

# Convert columns
df["date"] = pd.to_datetime(
    df["date"],
    errors="coerce"
)

df["position"] = pd.to_numeric(
    df["position"],
    errors="coerce"
)

df["popularity"] = pd.to_numeric(
    df["popularity"],
    errors="coerce"
)

df["duration_ms"] = pd.to_numeric(
    df["duration_ms"],
    errors="coerce"
)

df["total_tracks"] = pd.to_numeric(
    df["total_tracks"],
    errors="coerce"
)

# Clean text
df["song"] = (
    df["song"]
    .astype(str)
    .str.strip()
)

df["artist"] = (
    df["artist"]
    .astype(str)
    .str.strip()
)

df["album_type"] = (
    df["album_type"]
    .astype(str)
    .str.strip()
    .str.title()
)

# Normalize explicit values
df["is_explicit"] = (
    df["is_explicit"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin([
        "true",
        "1",
        "yes",
        "y",
        "explicit"
    ])
)

# Remove unusable rows
df = df.dropna(
    subset=[
        "date",
        "position",
        "song",
        "artist"
    ]
)

df["position"] = df["position"].astype(int)

raw_rows = len(df)

# Remove duplicate playlist positions
df = (
    df
    .sort_values(
        [
            "date",
            "position",
            "song",
            "artist"
        ]
    )
    .drop_duplicates(
        [
            "date",
            "position"
        ],
        keep="first"
    )
    .sort_values(
        [
            "date",
            "position"
        ]
    )
    .reset_index(drop=True)
)

# Additional features
df["duration_min"] = (
    df["duration_ms"] / 60000
)

df["song_id"] = (
    df["song"] +
    " — " +
    df["artist"]
)

print("STEP 5 COMPLETE")

print("Clean shape:", df.shape)

print(
    "Rows removed during "
    "date-position deduplication:",
    raw_rows - len(df)
)

print(
    "Unique dates:",
    df["date"].nunique()
)

print(
    "Date range:",
    df["date"].min().date(),
    "to",
    df["date"].max().date()
)

display(df.head())

STEP 5 COMPLETE
Clean shape: (27750, 12)
Rows removed during date-position deduplication: 50
Unique dates: 555
Date range: 2024-05-18 to 2025-11-27


/tmp/ipykernel_1460/1094932764.py:4: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["date"] = pd.to_datetime(


,date,position,song,artist,popularity,duration_ms,album_type,total_tracks,is_explicit,album_cover_url,duration_min,song_id
0,2024-05-18,1,Like Crazy,Jimin,93,212241,Single,6,False,https://i.scdn.co/image/ab67616d0000b2732b4607...,3.537350,Like Crazy — Jimin
1,2024-05-18,2,UNFORGIVEN (feat. Nile Rodgers),LE SSERAFIM,87,182148,Album,13,False,https://i.scdn.co/image/ab67616d0000b273d71fd7...,3.035800,UNFORGIVEN (feat. Nile Rodgers) — LE SSERAFIM
2,2024-05-18,3,Spicy,aespa,81,197040,Single,6,False,https://i.scdn.co/image/ab67616d0000b27304878a...,3.284000,Spicy — aespa
3,2024-05-18,4,I AM,IVE,89,183853,Album,11,False,https://i.scdn.co/image/ab67616d0000b27325ef3c...,3.064217,I AM — IVE
4,2024-05-18,5,Queencard,(G)I-DLE,71,161240,Single,6,False,https://i.scdn.co/image/ab67616d0000b27382dd24...,2.687333,Queencard — (G)I-DLE


In [6]:
daily_counts = (
    df.groupby("date")
    .size()
    .reset_index(name="records")
)

valid_days = daily_counts[
    daily_counts["records"] == 50
]

invalid_days = daily_counts[
    daily_counts["records"] != 50
]

print("STEP 6 COMPLETE")

print(
    "Total dates:",
    len(daily_counts)
)

print(
    "Valid 50-entry dates:",
    len(valid_days)
)

print(
    "Non-50-entry dates:",
    len(invalid_days)
)

print(
    "Clean rows:",
    len(df)
)

if len(invalid_days):
    print("\nNon-standard dates:")
    display(invalid_days.head(20))
else:
    print(
        "Every date has exactly 50 records."
    )

STEP 6 COMPLETE
Total dates: 555
Valid 50-entry dates: 555
Non-50-entry dates: 0
Clean rows: 27750
Every date has exactly 50 records.


In [8]:
work = df.copy()

# Previous appearance of each song
work["prev_date"] = (
    work
    .groupby("song_id")["date"]
    .shift(1)
)

work["prev_position"] = (
    work
    .groupby("song_id")["position"]
    .shift(1)
)

work["prev_popularity"] = (
    work
    .groupby("song_id")["popularity"]
    .shift(1)
)

# Gap between appearances
work["gap_days"] = (
    work["date"] -
    work["prev_date"]
).dt.days

# Re-entry if song was absent for at least one day
work["is_reentry"] = (
    work["gap_days"]
    .fillna(0) > 1
)

# Entry classification
work["entry_type"] = np.where(
    work["prev_date"].isna(),
    "First Entry",
    np.where(
        work["is_reentry"],
        "Re-entry",
        "Continuing"
    )
)

# Rank improvement
work["rank_jump"] = (
    work["prev_position"] -
    work["position"]
)

# Popularity change
work["popularity_change"] = (
    work["popularity"] -
    work["prev_popularity"]
)

print("STEP 7 COMPLETE")

print("\nEntry type counts:")

display(
    work["entry_type"]
    .value_counts()
    .to_frame("records")
)

print("\nSample re-entry events:")

display(
    work[
        work["is_reentry"]
    ][
        [
            "date",
            "song_id",
            "position",
            "prev_position",
            "gap_days",
            "rank_jump",
            "popularity_change"
        ]
    ].head(20)
)

STEP 7 COMPLETE

Entry type counts:


,records
entry_type,
Continuing,26134
Re-entry,1075
First Entry,541



Sample re-entry events:


,date,song_id,position,prev_position,gap_days,rank_jump,popularity_change
195,2024-05-21,Kill Bill — SZA,46,46.0,2.0,0.0,0.0
196,2024-05-21,Teddy Bear — STAYC,47,47.0,3.0,0.0,0.0
198,2024-05-21,LOVE DIVE — IVE,49,48.0,2.0,-1.0,0.0
296,2024-05-23,Love Always Run Away — Lim Young Woong,47,50.0,5.0,3.0,0.0
342,2024-05-24,VIBE (feat. Jimin of BTS) — TAEYANG,43,49.0,2.0,6.0,0.0
347,2024-05-24,Interlude : Dive — Jimin,48,38.0,4.0,-10.0,-1.0
399,2024-05-25,Love Always Run Away — Lim Young Woong,50,47.0,2.0,-3.0,0.0
542,2024-05-28,Like Crazy (Deep House Remix) — Jimin,43,46.0,8.0,3.0,-1.0
543,2024-05-28,Like Crazy (UK Garage Remix) — Jimin,44,47.0,8.0,3.0,-1.0
544,2024-05-28,Promise — Jimin,45,43.0,8.0,-2.0,-1.0


In [10]:
# ============================================================
# STEP 8 — MOMENTUM, RETENTION, RECOVERY & FANDOM KPIs
# ============================================================

print("STEP 8 STARTED...")

# Make a fresh copy
work = work.copy()

# ------------------------------------------------------------
# 1. MOMENTUM SPIKE SCORE
# ------------------------------------------------------------

# Rank improvement
# Example:
# Previous rank = 40
# New rank = 10
# Rank jump = 30
work["rank_jump"] = (
    work["prev_position"] -
    work["position"]
)

# Popularity change
work["popularity_change"] = (
    work["popularity"] -
    work["prev_popularity"]
)

# Rank component
rank_component = np.clip(
    (
        work["rank_jump"].fillna(0) + 50
    ) / 100 * 100,
    0,
    100
)

# Popularity component
pop_component = np.clip(
    (
        work["popularity_change"].fillna(0) + 20
    ) / 40 * 100,
    0,
    100
)

# Gap component
gap_component = np.clip(
    work["gap_days"].fillna(0) / 30 * 100,
    0,
    100
)

# Final Momentum Spike Score
work["momentum_spike_score"] = (
    0.50 * rank_component
    + 0.35 * pop_component
    + 0.15 * gap_component
)

print("✓ Momentum Spike Score calculated")


# ------------------------------------------------------------
# 2. CALCULATE RETENTION, PEAK RANK & RECOVERY SPEED
# ------------------------------------------------------------

peak_rank_values = {}
retention_values = {}
recovery_values = {}

for song_id in work["song_id"].dropna().unique():

    # Get this song's complete history
    song_data = (
        work[
            work["song_id"] == song_id
        ]
        .sort_values("date")
    )

    # Loop through each entry/re-entry
    for current_index, row in song_data.iterrows():

        if row["entry_type"] not in [
            "First Entry",
            "Re-entry"
        ]:
            continue

        start_date = row["date"]

        # ----------------------------------------------------
        # 7-DAY WINDOW
        # ----------------------------------------------------

        seven_day_data = song_data[
            (
                song_data["date"] >= start_date
            )
            &
            (
                song_data["date"]
                <= start_date +
                pd.Timedelta(days=7)
            )
        ]

        if len(seven_day_data) > 0:

            peak_rank = int(
                seven_day_data["position"].min()
            )

        else:

            peak_rank = int(
                row["position"]
            )

        # ----------------------------------------------------
        # RETENTION DAYS
        # ----------------------------------------------------

        later_data = song_data[
            song_data["date"] > start_date
        ]

        if len(later_data) > 0:

            last_date = later_data["date"].max()

            retention_days = int(
                (
                    last_date -
                    start_date
                ).days
            )

        else:

            retention_days = 0

        # ----------------------------------------------------
        # RANK RECOVERY SPEED
        # ----------------------------------------------------

        if (
            row["entry_type"] == "Re-entry"
            and pd.notna(row["gap_days"])
            and row["gap_days"] > 0
        ):

            rank_jump_value = max(
                float(row["rank_jump"]),
                0
            )

            recovery_speed = (
                rank_jump_value /
                float(row["gap_days"])
            )

        else:

            recovery_speed = 0.0

        # Store results using the original dataframe index
        peak_rank_values[current_index] = peak_rank
        retention_values[current_index] = retention_days
        recovery_values[current_index] = recovery_speed


# ------------------------------------------------------------
# 3. ADD RESULTS BACK TO DATAFRAME
# ------------------------------------------------------------

work["peak_rank_7d"] = (
    work.index.map(
        peak_rank_values
    )
)

work["retention_days"] = (
    work.index.map(
        retention_values
    )
)

work["rank_recovery_speed"] = (
    work.index.map(
        recovery_values
    )
)

# Fill missing values
work["peak_rank_7d"] = (
    work["peak_rank_7d"]
    .fillna(work["position"])
)

work["retention_days"] = (
    work["retention_days"]
    .fillna(0)
)

work["rank_recovery_speed"] = (
    work["rank_recovery_speed"]
    .fillna(0)
)

print("✓ Peak rank calculated")
print("✓ Retention calculated")
print("✓ Rank recovery speed calculated")


# ------------------------------------------------------------
# 4. RE-ENTRY FREQUENCY
# ------------------------------------------------------------

work["reentry_count"] = (
    work
    .groupby("song_id")["is_reentry"]
    .transform("sum")
)

print("✓ Re-entry frequency calculated")


# ------------------------------------------------------------
# 5. FANDOM INTENSITY PROXY
# ------------------------------------------------------------

# Re-entry frequency component
reentry_component = np.clip(
    work["reentry_count"]
    .fillna(0)
    * 20,
    0,
    100
)

# Recovery component
recovery_component = np.clip(
    work["rank_recovery_speed"]
    .fillna(0)
    * 10,
    0,
    100
)

# Retention component
retention_component = np.clip(
    work["retention_days"]
    .fillna(0)
    / 30
    * 100,
    0,
    100
)

# Fandom Intensity Proxy Score
work["fandom_intensity_proxy"] = (
    0.40 * reentry_component
    + 0.30 * np.clip(
        work["momentum_spike_score"],
        0,
        100
    )
    + 0.15 * recovery_component
    + 0.15 * retention_component
)

print("✓ Fandom Intensity Proxy calculated")


# ------------------------------------------------------------
# 6. CREATE COMEBACK DATAFRAME
# ------------------------------------------------------------

comeback = work[
    work["entry_type"] == "Re-entry"
].copy()


# ------------------------------------------------------------
# 7. DISPLAY RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("STEP 8 COMPLETE")
print("=" * 60)

print(
    "Total records:",
    len(work)
)

print(
    "Total re-entry events:",
    len(comeback)
)

if len(comeback) > 0:

    print(
        "Average Momentum Spike Score:",
        round(
            comeback[
                "momentum_spike_score"
            ].mean(),
            2
        )
    )

    print(
        "Average Retention Days:",
        round(
            comeback[
                "retention_days"
            ].mean(),
            2
        )
    )

    print(
        "Average Rank Recovery Speed:",
        round(
            comeback[
                "rank_recovery_speed"
            ].mean(),
            2
        )
    )

    print("\nTop 20 Comeback Events:")

    display(
        comeback[
            [
                "date",
                "song_id",
                "position",
                "prev_position",
                "gap_days",
                "rank_jump",
                "momentum_spike_score",
                "peak_rank_7d",
                "retention_days",
                "rank_recovery_speed",
                "fandom_intensity_proxy"
            ]
        ]
        .sort_values(
            "momentum_spike_score",
            ascending=False
        )
        .head(20)
    )

else:

    print(
        "⚠️ No re-entry events were detected."
    )

print("\n✓ STEP 8 FINISHED SUCCESSFULLY")

STEP 8 STARTED...
✓ Momentum Spike Score calculated
✓ Peak rank calculated
✓ Retention calculated
✓ Rank recovery speed calculated
✓ Re-entry frequency calculated
✓ Fandom Intensity Proxy calculated

STEP 8 COMPLETE
Total records: 27750
Total re-entry events: 1075
Average Momentum Spike Score: 45.2
Average Retention Days: 81.06
Average Rank Recovery Speed: 0.49

Top 20 Comeback Events:


,date,song_id,position,prev_position,gap_days,rank_jump,momentum_spike_score,peak_rank_7d,retention_days,rank_recovery_speed,fandom_intensity_proxy
12157,2025-01-16,Lost and Found — KingLee & Lil' Flip,8,27.0,19.0,19.0,75.500,8.0,1.0,1.000000,32.650000
8577,2024-11-05,Christmas Love — Jimin,28,48.0,136.0,20.0,68.375,28.0,53.0,0.147059,75.733088
24431,2025-09-22,Who (Beautiful Mind Remix) — Jimin,32,49.0,30.0,17.0,66.875,32.0,66.0,0.566667,75.912500
14648,2025-03-06,STUPID IN LOVE (feat. HUH YUNJIN of LE SSERAFI...,49,46.0,16.0,-3.0,66.500,45.0,45.0,0.000000,58.950000
24429,2025-09-22,Who (Instrumental) — Jimin,30,45.0,29.0,15.0,66.250,30.0,66.0,0.517241,75.650862
24430,2025-09-22,Who (Shibuyakei Remix) — Jimin,31,45.0,30.0,14.0,66.250,31.0,66.0,0.466667,75.575000
12527,2025-01-23,With you — Jimin & HA SUNG WOON,28,50.0,201.0,22.0,65.875,28.0,0.0,0.109453,51.926679
25135,2025-10-06,Armageddon — aespa & Grimes & Flava D,36,31.0,15.0,-5.0,65.000,36.0,0.0,0.000000,27.500000
22341,2025-08-10,Perfect Night — LE SSERAFIM,42,50.0,51.0,8.0,64.125,36.0,72.0,0.156863,74.472794
12225,2025-01-17,All Or Nothing — J Money & KingLee,26,30.0,26.0,4.0,63.625,26.0,0.0,0.153846,27.318269



✓ STEP 8 FINISHED SUCCESSFULLY


In [11]:
import plotly.express as px

comeback = work[
    work["entry_type"] == "Re-entry"
].copy()

print("STEP 9 — TOP COMEBACKS")

top_comebacks = (
    comeback
    .sort_values(
        "momentum_spike_score",
        ascending=False
    )
    [
        [
            "date",
            "song_id",
            "position",
            "prev_position",
            "gap_days",
            "momentum_spike_score",
            "peak_rank_7d",
            "retention_days"
        ]
    ]
    .head(15)
)

display(top_comebacks)


print(
    "STEP 9 — FANDOM INTENSITY LEADERBOARD"
)

leaderboard = (
    work
    .groupby(
        ["song_id", "artist"],
        as_index=False
    )
    .agg(
        re_entries=(
            "is_reentry",
            "sum"
        ),
        avg_momentum=(
            "momentum_spike_score",
            "mean"
        ),
        avg_retention=(
            "retention_days",
            "mean"
        ),
        avg_recovery_speed=(
            "rank_recovery_speed",
            "mean"
        ),
        fandom_intensity_proxy=(
            "fandom_intensity_proxy",
            "mean"
        )
    )
    .sort_values(
        "fandom_intensity_proxy",
        ascending=False
    )
    .head(15)
)

display(leaderboard)


fig = px.bar(
    leaderboard.sort_values(
        "fandom_intensity_proxy"
    ),
    x="fandom_intensity_proxy",
    y="song_id",
    orientation="h",
    title=(
        "Top Songs by "
        "Fandom Intensity Proxy"
    )
)

fig.show()

STEP 9 — TOP COMEBACKS


,date,song_id,position,prev_position,gap_days,momentum_spike_score,peak_rank_7d,retention_days
12157,2025-01-16,Lost and Found — KingLee & Lil' Flip,8,27.0,19.0,75.500,8.0,1.0
8577,2024-11-05,Christmas Love — Jimin,28,48.0,136.0,68.375,28.0,53.0
24431,2025-09-22,Who (Beautiful Mind Remix) — Jimin,32,49.0,30.0,66.875,32.0,66.0
14648,2025-03-06,STUPID IN LOVE (feat. HUH YUNJIN of LE SSERAFI...,49,46.0,16.0,66.500,45.0,45.0
24429,2025-09-22,Who (Instrumental) — Jimin,30,45.0,29.0,66.250,30.0,66.0
24430,2025-09-22,Who (Shibuyakei Remix) — Jimin,31,45.0,30.0,66.250,31.0,66.0
12527,2025-01-23,With you — Jimin & HA SUNG WOON,28,50.0,201.0,65.875,28.0,0.0
25135,2025-10-06,Armageddon — aespa & Grimes & Flava D,36,31.0,15.0,65.000,36.0,0.0
22341,2025-08-10,Perfect Night — LE SSERAFIM,42,50.0,51.0,64.125,36.0,72.0
12225,2025-01-17,All Or Nothing — J Money & KingLee,26,30.0,26.0,63.625,26.0,0.0


STEP 9 — FANDOM INTENSITY LEADERBOARD


,song_id,artist,re_entries,avg_momentum,avg_retention,avg_recovery_speed,fandom_intensity_proxy
73,Christmas Love — Jimin,Jimin,14,44.077586,26.793103,0.322887,60.138641
245,Like Crazy (Deep House Remix) — Jimin,Jimin,9,43.334821,9.571429,0.115774,57.031250
248,Like Crazy (UK Garage Remix) — Jimin,Jimin,8,43.310000,8.400000,0.175000,56.995500
454,Talk Saxy — RIIZE,RIIZE,6,45.698864,11.681818,0.077094,56.825301
80,Closer to You (feat. Major Lazer) — Jung Kook,Jung Kook,13,43.735417,31.283333,0.037902,56.427477
431,Sunflower - Spider-Man: Into the Spider-Verse ...,Post Malone,5,43.750000,16.652174,0.017391,56.411957
410,Somebody — Jung Kook,Jung Kook,15,43.493952,12.919355,0.048387,56.257863
511,You Were Beautiful — DAY6,DAY6,13,43.552083,17.466667,0.172811,56.199842
336,Promise — Jimin,Jimin,9,43.246429,7.914286,0.114286,56.116786
298,My You — Jung Kook,Jung Kook,6,43.316667,12.766667,0.050000,56.086667


In [12]:
# ============================================================
# STEP 10 — CREATE STREAMLIT DASHBOARD
# ============================================================

print("STEP 10 STARTED...")

app_code = r'''
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px

# ============================================================
# PAGE CONFIGURATION
# ============================================================

st.set_page_config(
    page_title="Comeback Momentum Analytics",
    page_icon="🎵",
    layout="wide"
)

# ============================================================
# TITLE
# ============================================================

st.title(
    "🎵 Comeback Momentum, Chart Re-Entry & "
    "Fandom Intensity Analysis"
)

st.markdown(
    """
    ### South Korea Top 50 Playlist Analytics

    This interactive dashboard analyzes chart re-entries,
    comeback momentum, retention, rank recovery and a
    **Fandom Intensity Proxy** based on observable playlist data.
    """
)

# ============================================================
# LOAD DATA
# ============================================================

@st.cache_data
def load_data():

    file_path = "Atlantic_South_Korea.csv"

    df = pd.read_csv(file_path)

    df["date"] = pd.to_datetime(df["date"])

    return df


try:

    df = load_data()

except Exception as e:

    st.error(
        "❌ Dataset could not be loaded."
    )

    st.info(
        "Make sure Atlantic_South_Korea.csv "
        "is in the same folder as this Streamlit app."
    )

    st.stop()


# ============================================================
# BASIC CLEANING
# ============================================================

df = df.copy()

if "song_id" not in df.columns:

    df["song_id"] = (
        df["song"].astype(str).str.strip()
        + " — "
        + df["artist"].astype(str).str.strip()
    )

if "entry_type" not in df.columns:

    df = df.sort_values(
        ["song_id", "date"]
    )

    df["prev_date"] = (
        df.groupby("song_id")["date"]
        .shift(1)
    )

    df["gap_days"] = (
        df["date"] -
        df["prev_date"]
    ).dt.days

    df["entry_type"] = np.where(
        df["prev_date"].isna(),
        "First Entry",
        np.where(
            df["gap_days"] > 1,
            "Re-entry",
            "Continuing"
        )
    )

if "is_reentry" not in df.columns:

    df["is_reentry"] = (
        df["entry_type"] == "Re-entry"
    ).astype(int)


# ============================================================
# SIDEBAR FILTERS
# ============================================================

st.sidebar.header("🎛️ Dashboard Filters")

min_date = df["date"].min().date()
max_date = df["date"].max().date()

date_range = st.sidebar.date_input(
    "Select Date Range",
    value=(min_date, max_date),
    min_value=min_date,
    max_value=max_date
)

if isinstance(date_range, tuple) and len(date_range) == 2:

    start_date = pd.Timestamp(date_range[0])
    end_date = pd.Timestamp(date_range[1])

else:

    start_date = pd.Timestamp(min_date)
    end_date = pd.Timestamp(max_date)


# Artist filter
artists = sorted(
    df["artist"]
    .dropna()
    .astype(str)
    .unique()
)

selected_artists = st.sidebar.multiselect(
    "Select Artist",
    artists
)


# Song filter
songs = sorted(
    df["song"]
    .dropna()
    .astype(str)
    .unique()
)

selected_songs = st.sidebar.multiselect(
    "Select Song",
    songs
)


# Album type filter
if "album_type" in df.columns:

    album_types = sorted(
        df["album_type"]
        .dropna()
        .astype(str)
        .unique()
    )

    selected_album_types = st.sidebar.multiselect(
        "Album Type",
        album_types
    )

else:

    selected_album_types = []


# Minimum re-entry filter
min_reentries = st.sidebar.slider(
    "Minimum Re-entry Count",
    min_value=0,
    max_value=10,
    value=0
)


# ============================================================
# APPLY FILTERS
# ============================================================

filtered = df[
    (df["date"] >= start_date)
    &
    (df["date"] <= end_date)
].copy()


if selected_artists:

    filtered = filtered[
        filtered["artist"].isin(
            selected_artists
        )
    ]


if selected_songs:

    filtered = filtered[
        filtered["song"].isin(
            selected_songs
        )
    ]


if selected_album_types:

    filtered = filtered[
        filtered["album_type"].isin(
            selected_album_types
        )
    ]


# ============================================================
# RE-ENTRY COUNTS
# ============================================================

reentry_counts = (
    filtered.groupby("song_id")["is_reentry"]
    .sum()
    .reset_index(name="reentry_count")
)

filtered = filtered.merge(
    reentry_counts,
    on="song_id",
    how="left"
)

filtered = filtered[
    filtered["reentry_count"] >= min_reentries
]


# ============================================================
# DATA CHECK
# ============================================================

if filtered.empty:

    st.warning(
        "⚠️ No data matches the selected filters."
    )

    st.stop()


# ============================================================
# KPI CALCULATIONS
# ============================================================

comeback = filtered[
    filtered["entry_type"] == "Re-entry"
].copy()


total_songs = filtered["song_id"].nunique()

total_records = len(filtered)

total_reentries = len(comeback)

avg_momentum = (
    comeback["momentum_spike_score"].mean()
    if "momentum_spike_score" in comeback.columns
    and len(comeback) > 0
    else 0
)

avg_retention = (
    comeback["retention_days"].mean()
    if "retention_days" in comeback.columns
    and len(comeback) > 0
    else 0
)

avg_recovery = (
    comeback["rank_recovery_speed"].mean()
    if "rank_recovery_speed" in comeback.columns
    and len(comeback) > 0
    else 0
)


# ============================================================
# KPI CARDS
# ============================================================

st.subheader("📊 Key Performance Indicators")

col1, col2, col3, col4, col5 = st.columns(5)

col1.metric(
    "Songs",
    total_songs
)

col2.metric(
    "Records",
    f"{total_records:,}"
)

col3.metric(
    "Re-entry Events",
    total_reentries
)

col4.metric(
    "Avg Momentum",
    f"{avg_momentum:.2f}"
)

col5.metric(
    "Avg Retention",
    f"{avg_retention:.1f} days"
)


# ============================================================
# TABS
# ============================================================

tab1, tab2, tab3, tab4, tab5, tab6 = st.tabs(
    [
        "🏠 Overview",
        "🔄 Re-entry Explorer",
        "🚀 Momentum",
        "💿 Content Attributes",
        "🔥 Fandom Proxy",
        "🔍 Data Quality"
    ]
)


# ============================================================
# TAB 1 — OVERVIEW
# ============================================================

with tab1:

    st.header(
        "Overview of Chart Performance"
    )

    daily_entries = (
        filtered.groupby("date")
        .size()
        .reset_index(
            name="entries"
        )
    )

    fig = px.line(
        daily_entries,
        x="date",
        y="entries",
        title="Daily Playlist Entries"
    )

    st.plotly_chart(
        fig,
        use_container_width=True
    )


    # Top artists

    st.subheader(
        "Top Artists by Chart Entries"
    )

    artist_counts = (
        filtered.groupby("artist")
        .size()
        .reset_index(
            name="entries"
        )
        .sort_values(
            "entries",
            ascending=False
        )
        .head(15)
    )

    fig_artist = px.bar(
        artist_counts,
        x="entries",
        y="artist",
        orientation="h",
        title="Top 15 Artists"
    )

    st.plotly_chart(
        fig_artist,
        use_container_width=True
    )


# ============================================================
# TAB 2 — RE-ENTRY EXPLORER
# ============================================================

with tab2:

    st.header(
        "🔄 Chart Re-entry Explorer"
    )

    if len(comeback) == 0:

        st.warning(
            "No re-entry events found."
        )

    else:

        st.metric(
            "Total Re-entry Events",
            len(comeback)
        )


        # Re-entry timeline

        timeline = comeback.sort_values(
            "date"
        )

        fig = px.scatter(
            timeline,
            x="date",
            y="position",
            color="artist",
            hover_data=[
                "song",
                "artist",
                "gap_days",
                "position"
            ],
            title="Re-entry Timeline"
        )

        fig.update_yaxes(
            autorange="reversed"
        )

        st.plotly_chart(
            fig,
            use_container_width=True
        )


        # Re-entry leaderboard

        st.subheader(
            "Songs with Most Re-entries"
        )

        leaderboard = (
            comeback.groupby(
                ["song", "artist"]
            )
            .size()
            .reset_index(
                name="reentry_events"
            )
            .sort_values(
                "reentry_events",
                ascending=False
            )
            .head(20)
        )

        st.dataframe(
            leaderboard,
            use_container_width=True
        )


# ============================================================
# TAB 3 — MOMENTUM
# ============================================================

with tab3:

    st.header(
        "🚀 Comeback Momentum Analysis"
    )

    if len(comeback) == 0:

        st.warning(
            "No comeback events available."
        )

    else:

        # Momentum distribution

        fig = px.histogram(
            comeback,
            x="momentum_spike_score",
            nbins=20,
            title="Momentum Spike Score Distribution"
        )

        st.plotly_chart(
            fig,
            use_container_width=True
        )


        # Momentum vs retention

        if "retention_days" in comeback.columns:

            fig = px.scatter(
                comeback,
                x="momentum_spike_score",
                y="retention_days",
                color="artist",
                hover_data=[
                    "song",
                    "artist",
                    "position"
                ],
                title="Momentum vs Post-Comeback Retention"
            )

            st.plotly_chart(
                fig,
                use_container_width=True
            )


        # Top comeback events

        st.subheader(
            "🔥 Strongest Comeback Events"
        )

        momentum_table = (
            comeback[
                [
                    "date",
                    "song",
                    "artist",
                    "position",
                    "gap_days",
                    "momentum_spike_score",
                    "retention_days"
                ]
            ]
            .sort_values(
                "momentum_spike_score",
                ascending=False
            )
            .head(20)
        )

        st.dataframe(
            momentum_table,
            use_container_width=True
        )


# ============================================================
# TAB 4 — CONTENT ATTRIBUTES
# ============================================================

with tab4:

    st.header(
        "💿 Content Attributes vs Momentum"
    )

    if "album_type" in filtered.columns:

        album_analysis = (
            filtered.groupby("album_type")
            .agg(
                average_popularity=(
                    "popularity",
                    "mean"
                ),
                average_position=(
                    "position",
                    "mean"
                ),
                entries=(
                    "song_id",
                    "count"
                )
            )
            .reset_index()
        )

        fig = px.bar(
            album_analysis,
            x="album_type",
            y="average_popularity",
            title="Average Popularity by Album Type"
        )

        st.plotly_chart(
            fig,
            use_container_width=True
        )


    # Explicit vs clean

    if "is_explicit" in filtered.columns:

        explicit_analysis = (
            filtered.groupby("is_explicit")
            .size()
            .reset_index(
                name="entries"
            )
        )

        explicit_analysis[
            "content_type"
        ] = explicit_analysis[
            "is_explicit"
        ].map(
            {
                True: "Explicit",
                False: "Clean",
                1: "Explicit",
                0: "Clean"
            }
        )

        fig = px.pie(
            explicit_analysis,
            names="content_type",
            values="entries",
            title="Explicit vs Clean Content"
        )

        st.plotly_chart(
            fig,
            use_container_width=True
        )


# ============================================================
# TAB 5 — FANDOM INTENSITY PROXY
# ============================================================

with tab5:

    st.header(
        "🔥 Fandom Intensity Proxy"
    )

    st.info(
        "The Fandom Intensity Proxy is a constructed analytical "
        "proxy based on observable chart behavior. It does not "
        "directly measure real-world fan activity."
    )

    if (
        "fandom_intensity_proxy" in filtered.columns
    ):

        fandom = (
            filtered.groupby(
                ["song", "artist"]
            )
            .agg(
                fandom_intensity=(
                    "fandom_intensity_proxy",
                    "max"
                ),
                reentries=(
                    "is_reentry",
                    "sum"
                ),
                avg_momentum=(
                    "momentum_spike_score",
                    "mean"
                )
            )
            .reset_index()
            .sort_values(
                "fandom_intensity",
                ascending=False
            )
            .head(20)
        )


        fig = px.bar(
            fandom,
            x="fandom_intensity",
            y="song",
            color="artist",
            orientation="h",
            title="Fandom Intensity Proxy Leaderboard"
        )

        st.plotly_chart(
            fig,
            use_container_width=True
        )


        st.subheader(
            "Fandom Intensity Details"
        )

        st.dataframe(
            fandom,
            use_container_width=True
        )


# ============================================================
# TAB 6 — DATA QUALITY
# ============================================================

with tab6:

    st.header(
        "🔍 Data Quality & Validation"
    )

    col1, col2, col3 = st.columns(3)

    col1.metric(
        "Rows",
        len(filtered)
    )

    col2.metric(
        "Columns",
        len(filtered.columns)
    )

    col3.metric(
        "Unique Songs",
        filtered["song_id"].nunique()
    )


    st.subheader(
        "Missing Values"
    )

    missing = (
        filtered.isnull()
        .sum()
        .reset_index()
    )

    missing.columns = [
        "column",
        "missing_values"
    ]

    missing = missing[
        missing["missing_values"] > 0
    ]

    if len(missing) > 0:

        st.dataframe(
            missing,
            use_container_width=True
        )

    else:

        st.success(
            "✅ No missing values detected."
        )


    st.subheader(
        "Dataset Preview"
    )

    st.dataframe(
        filtered.head(100),
        use_container_width=True
    )


# ============================================================
# FOOTER
# ============================================================

st.markdown("---")

st.caption(
    "Comeback Momentum, Chart Re-Entry, and Fandom Intensity "
    "Analysis of South Korea Top 50 Playlist"
)

st.caption(
    "Analytical dashboard created using Python, Pandas, "
    "Plotly and Streamlit."
)
'''

with open(
    "comeback_analysis_app.py",
    "w",
    encoding="utf-8"
) as f:

    f.write(app_code)


print("✓ Streamlit app code created")
print("✓ File: comeback_analysis_app.py")
print("✓ Dashboard contains 6 interactive sections")
print("\nSTEP 10 FINISHED SUCCESSFULLY")

STEP 10 STARTED...
✓ Streamlit app code created
✓ File: comeback_analysis_app.py
✓ Dashboard contains 6 interactive sections

STEP 10 FINISHED SUCCESSFULLY


In [13]:
import subprocess
import time
import os

# Stop old Streamlit processes if notebook is rerun
subprocess.run(
    ["pkill", "-f", "streamlit"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(2)

process = subprocess.Popen(
    [
        "streamlit",
        "run",
        "comeback_analysis_app.py",
        "--server.port",
        "8501",
        "--server.address",
        "0.0.0.0",
        "--server.headless",
        "true"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(5)

print("STEP 11 COMPLETE")
print("Streamlit server started.")
print("Port: 8501")

STEP 11 COMPLETE
Streamlit server started.
Port: 8501


In [17]:
# ============================================================
# STEP 12 — OPEN STREAMLIT DASHBOARD IN COLAB
# ============================================================

print("STEP 12 STARTED...")

import subprocess
import time
import os

# Stop old Streamlit processes
os.system("pkill -f streamlit || true")

# Start Streamlit
process = subprocess.Popen(
    [
        "streamlit",
        "run",
        "comeback_analysis_app.py",
        "--server.port=8501",
        "--server.address=0.0.0.0",
        "--server.headless=true"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

# Wait for Streamlit
time.sleep(8)

print("✓ Streamlit server started")
print()
print("Opening dashboard...")
print()

from google.colab import output

# Recommended Colab method
output.serve_kernel_port_as_iframe(8501)

print()
print("✓ STEP 12 FINISHED")

STEP 12 STARTED...
✓ Streamlit server started

Opening dashboard...



<IPython.core.display.Javascript object>


✓ STEP 12 FINISHED
